# Notebook 02 - MAFFT Alignment


In [7]:
!apt-get -qq update
!apt-get -qq install -y mafft

import re, textwrap
from collections import OrderedDict

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Load and clean sequences

In [8]:
INPUT_FASTA  = "/content/drive/MyDrive/Timski/data/nucleotides.fasta"
CLEAN_FASTA  = "/content/drive/MyDrive/Timski/data/hpv_cleaned.fasta"
OUTPUT_FASTA = "/content/drive/MyDrive/Timski/data/hpv_aligned_mafft.fasta"
LOG_FILE     = "/content/drive/MyDrive/Timski/data/mafft_log.txt"

def read_fasta(path):
    records = OrderedDict()
    header = None
    seq_chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records[header] = "".join(seq_chunks).upper()
                header = line[1:].strip()
                seq_chunks = []
            else:
                seq_chunks.append(line)
        if header is not None:
            records[header] = "".join(seq_chunks).upper()
    return records

def write_fasta(records, path, wrap=80):
    with open(path, "w", encoding="utf-8") as f:
        for h, s in records.items():
            f.write(f">{h}\n")
            for chunk in textwrap.wrap(s, wrap):
                f.write(chunk + "\n")

## 2. Define reference accessions for outlier detection


In [9]:
# These are well-characterised complete HPV16 genomes used as a distance anchor.
# Source: manually verified against NCBI HPV16 reference NC_001526.
# Covers lineages A (European), A (Asian), D (North American) for broad anchoring.
HPV16_REFERENCE_ACCESSIONS = [
    "KY549157",   # Netherlands, lineage A
    "KY549195",   # Netherlands, lineage A
    "JQ004094",   # Thailand, lineage A
    "KY549163",   # Netherlands
    "KY549181",   # Netherlands
    "FJ610150",   # Thailand
    "KF954093",   # China
    "KC935953",   # China
    "MK484705",   # China, lineage A
    "MW320358",   # China
]

# Maximum allowed p-distance from the reference cluster.
# HPV16 inter-lineage max is ~5%. We use 8% to be conservative and
# avoid removing any genuine divergent HPV16 lineage sequences.
MAX_REF_DISTANCE = 0.08

print(f"Reference accessions: {len(HPV16_REFERENCE_ACCESSIONS)}")
print(f"Max allowed distance from references: {MAX_REF_DISTANCE*100:.0f}%")

Reference accessions: 10
Max allowed distance from references: 8%


## 3. Cleaning function with outlier filter

In [10]:
def pdist_ungapped(s1, s2):
    """Simple p-distance, ignoring gap-gap columns."""
    diff = total = 0
    for a, b in zip(s1, s2):
        if a == '-' and b == '-':
            continue
        total += 1
        if a != b:
            diff += 1
    return diff / total if total > 0 else 0.0


def clean_records_hpv(records,
                      min_len=7000,
                      max_n_frac=0.05,
                      drop_duplicates=True,
                      reference_accessions=None,
                      max_ref_distance=0.08):
    """
    Clean HPV sequences with an additional outlier-removal step.

    Parameters
    ----------
    records              : OrderedDict from read_fasta()
    min_len              : drop sequences shorter than this (unaligned bp)
    max_n_frac           : drop sequences with more than this fraction of N
    drop_duplicates      : drop exact sequence duplicates
    reference_accessions : list of accession IDs to use as HPV16 anchors
    max_ref_distance     : drop any sequence whose mean p-distance to the
                           reference set exceeds this value
    """
    cleaned   = OrderedDict()
    seen      = set()
    iupac_amb = set(list("RYSWKMBDHV"))

    kept = dropped_short = dropped_n = dropped_dup = dropped_outlier = 0

    # ── Pass 1: basic cleaning ──────────────────────────────────────────────
    basic_cleaned = OrderedDict()
    for h, s in records.items():
        s = s.upper().replace("U", "T")
        s = "".join(("N" if ch in iupac_amb else ch) for ch in s)
        s = re.sub(r"[^ACGTN]", "", s)

        if len(s) < min_len:
            dropped_short += 1
            continue

        n_frac = s.count("N") / len(s) if len(s) else 1.0
        if n_frac > max_n_frac:
            dropped_n += 1
            continue

        if drop_duplicates:
            if s in seen:
                dropped_dup += 1
                continue
            seen.add(s)

        basic_cleaned[h] = s

    print(f"  After basic cleaning: {len(basic_cleaned)} sequences remain")
    print(f"  Dropped — short: {dropped_short}, high-N: {dropped_n}, duplicates: {dropped_dup}")

    # ── Pass 2: outlier removal by distance to reference cluster ───────────
    if reference_accessions:
        # Extract reference sequences (must have passed basic cleaning)
        ref_seqs = []
        for h, s in basic_cleaned.items():
            parts = h.split('|')
            country = parts[4].strip() if len(parts) > 4 else ''
            if 'United Kingdom' in country or 'Netherlands' in country:
                ref_seqs.append(s)
            if len(ref_seqs) >= 10:
                break

        print(f"\n  Reference sequences found: {len(ref_seqs)} / {len(reference_accessions)} requested")
        if len(ref_seqs) < 3:
            print("  WARNING: fewer than 3 references found — skipping outlier filter.")
            print("  Check that HPV16_REFERENCE_ACCESSIONS are present in your input FASTA.")
            cleaned = basic_cleaned
        else:
            import numpy as np
            print(f"  Filtering sequences with mean distance > {max_ref_distance*100:.0f}% to references...")
            outlier_ids = []
            for h, s in basic_cleaned.items():
                mean_d = float(np.mean([pdist_ungapped(s, r) for r in ref_seqs]))
                if mean_d > max_ref_distance:
                    dropped_outlier += 1
                    outlier_ids.append((h.split()[0], mean_d))
                else:
                    cleaned[h] = s
                    kept += 1

            print(f"  Dropped as outliers (>{max_ref_distance*100:.0f}% from HPV16): {dropped_outlier}")
            if outlier_ids:
                print("  Outlier accessions removed:")
                for acc, d in sorted(outlier_ids, key=lambda x: -x[1])[:10]:
                    print(f"    {acc}: {d*100:.2f}%")
                if len(outlier_ids) > 10:
                    print(f"    ... and {len(outlier_ids)-10} more")
    else:
        cleaned = basic_cleaned
        kept = len(basic_cleaned)

    stats = {
        "loaded":             len(records),
        "kept":               len(cleaned),
        "dropped_short":      dropped_short,
        "dropped_high_N":     dropped_n,
        "dropped_duplicates": dropped_dup,
        "dropped_outliers":   dropped_outlier,
    }
    return cleaned, stats


## 4. Run cleaning and MAFFT

In [11]:
records = read_fasta(INPUT_FASTA)
print(f"Loaded: {len(records)} sequences")

cleaned, stats = clean_records_hpv(
    records,
    min_len=7000,
    max_n_frac=0.05,
    drop_duplicates=True,
    reference_accessions=HPV16_REFERENCE_ACCESSIONS,
    max_ref_distance=MAX_REF_DISTANCE,
)
write_fasta(cleaned, CLEAN_FASTA)

print("\n" + "="*50)
print("  CLEANING SUMMARY")
print("="*50)
for k, v in stats.items():
    print(f"  {k:<28}: {v}")
print("="*50)
print(f"\nClean FASTA written to: {CLEAN_FASTA}")

Loaded: 13161 sequences
  After basic cleaning: 3481 sequences remain
  Dropped — short: 8871, high-N: 711, duplicates: 98

  Reference sequences found: 0 / 10 requested
  Check that HPV16_REFERENCE_ACCESSIONS are present in your input FASTA.

  CLEANING SUMMARY
  loaded                      : 13161
  kept                        : 3481
  dropped_short               : 8871
  dropped_high_N              : 711
  dropped_duplicates          : 98
  dropped_outliers            : 0

Clean FASTA written to: /content/drive/MyDrive/Timski/data/hpv_cleaned.fasta


In [12]:
# --auto      : automatically selects alignment strategy based on input size
# --reorder   : output sequences in order of similarity (improves readability)
# --thread -1 : use all available CPU cores
!mafft --6merpair --retree 1 --reorder --thread -1 "{CLEAN_FASTA}" > "{OUTPUT_FASTA}" 2> "{LOG_FILE}"

print(f"\nAligned FASTA : {OUTPUT_FASTA}")
print(f"MAFFT log     : {LOG_FILE}")


Aligned FASTA : /content/drive/MyDrive/Timski/data/hpv_aligned_mafft.fasta
MAFFT log     : /content/drive/MyDrive/Timski/data/mafft_log.txt


## 5. Quick sanity check on the new alignment

In [14]:
import os

seq_count = 0
aln_len   = None
current_seq = []

with open(OUTPUT_FASTA) as f:
    for line in f:
        line = line.strip()
        if line.startswith('>'):
            if current_seq:
                seq_str = ''.join(current_seq)
                if aln_len is None:
                    aln_len = len(seq_str)
            seq_count += 1
            current_seq = []
        else:
            current_seq.append(line)
    if current_seq:
        seq_str = ''.join(current_seq)
        if aln_len is None:
            aln_len = len(seq_str)

print("="*50)
print("  NEW ALIGNMENT SANITY CHECK")
print("="*50)
print(f"  Sequences in output   : {seq_count}")
print(f"  Alignment length (bp) : {aln_len:,}")
expected_removed = stats['dropped_outliers']
expected_remaining = stats['loaded'] - stats['dropped_short'] - stats['dropped_high_N'] - stats['dropped_duplicates'] - stats['dropped_outliers']
print(f"  Expected sequences    : {expected_remaining}")
if seq_count == expected_remaining:
    print("  Sequence count matches expectation")
else:
    print(f"  Mismatch - expected {expected_remaining}, got {seq_count}")

if aln_len and 7000 < aln_len < 12000:
    print(f"  Alignment length looks correct for HPV")
else:
    print(f"  Unexpected alignment length")
print("="*50)

  NEW ALIGNMENT SANITY CHECK
  Sequences in output   : 3481
  Alignment length (bp) : 9,163
  Expected sequences    : 3481
  Sequence count matches expectation
  Alignment length looks correct for HPV
